# ДЗ 7. Часові ряди та напівструктуровані дані у задачах ML

**Дані:** NYC TLC Yellow Taxi Trip Records — [nyc.gov/site/tlc/about/tlc-trip-record-data.page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), місячні Parquet-файли за лютий і березень 2024.
JSONB event-log — синтетичний, генерується в notebook.

## 1. Setup і завантаження даних

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path
from urllib.request import urlretrieve

warnings.filterwarnings("ignore")

pkg = "pgserver" if sys.version_info < (3, 13) else "pixeltable-pgserver"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                       pkg, "jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "pyarrow", "requests"])

import pandas as pd
from sqlalchemy import create_engine, text

pgdata = Path("/tmp/hw7_pg")
if pkg == "pgserver":
    import pgserver
    pg = pgserver.get_server(pgdata, cleanup_mode="stop")
else:
    import pixeltable_pgserver
    pg = pixeltable_pgserver.get_server(pgdata, postgres_version=16)

engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 0
%config SqlMagic.feedback = 0
%sql engine

PostgreSQL 16.14 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


### NYC Taxi з офіційного Parquet

`head(500_000)` з одного місяця покриває лише перші 6 днів (файл упорядкований за часом), і тоді `trips_lag_30d` був би завжди `NULL`. Тому беремо два місяці — лютий і березень 2024 (60 днів) — і з кожного фіксовану випадкову вибірку 250 000 поїздок (`random_state=42`), разом 500 000. Поїздки з датою поза цими місяцями відкидаємо разом із базовим очищенням.

In [2]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
MONTHS = ["2024-02", "2024-03"]
columns = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
    "trip_distance", "fare_amount", "passenger_count",
]

data_dir = Path("/content" if Path("/content").exists() else "data")
data_dir.mkdir(parents=True, exist_ok=True)

parts = []
for month in MONTHS:
    parquet_path = data_dir / f"yellow_tripdata_{month}.parquet"
    if not parquet_path.exists():
        urlretrieve(BASE_URL + parquet_path.name, parquet_path)
    month_df = pd.read_parquet(parquet_path, columns=columns)
    parts.append(month_df.sample(n=250_000, random_state=42))
df = pd.concat(parts, ignore_index=True)

df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 500)]
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
df = df[df["PULocationID"].notna()]
df = df[(df["tpep_pickup_datetime"] >= "2024-02-01") & (df["tpep_pickup_datetime"] < "2024-04-01")]

df.to_sql("nyc_taxi_raw", engine, if_exists="replace", index=False, chunksize=10_000)
print(df.shape)

(481149, 7)


In [3]:
%%sql
SELECT COUNT(*) AS n_rows, MIN(tpep_pickup_datetime) AS min_pickup, MAX(tpep_pickup_datetime) AS max_pickup FROM nyc_taxi_raw;

n_rows,min_pickup,max_pickup
481149,2024-02-01 00:00:16,2024-03-31 23:59:24


### Daily aggregation `t7_hw_taxi_daily`

In [4]:
%%sql
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT "PULocationID"::INTEGER AS pickup_zone_id,
       DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
       COUNT(*)::INTEGER AS trip_count,
       AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
       SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0 AND fare_amount < 500
  AND trip_distance > 0 AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY "PULocationID"::INTEGER, DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily ADD PRIMARY KEY (pickup_zone_id, ts_day);

SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;

pickup_zone_id,n_days
1,10
2,1
3,10
4,60
6,2
7,60
9,8
10,57
11,22
12,56


In [5]:
%%sql
WITH zone_days AS (
    SELECT pickup_zone_id, COUNT(*) AS n_days
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
)
SELECT (SELECT COUNT(*) FROM t7_hw_taxi_daily) AS n_rows,
       COUNT(*) AS n_zones,
       COUNT(*) FILTER (WHERE n_days = 60) AS zones_with_full_60d,
       COUNT(*) FILTER (WHERE n_days < 31) AS zones_under_31d,
       (SELECT MIN(ts_day) FROM t7_hw_taxi_daily) AS first_day,
       (SELECT MAX(ts_day) FROM t7_hw_taxi_daily) AS last_day
FROM zone_days;

n_rows,n_zones,zones_with_full_60d,zones_under_31d,first_day,last_day
8486,240,66,109,2024-02-01,2024-03-31


**Interpretation:** З вибірки отримано 8 486 рядків `(pickup_zone_id, ts_day)` по 240 зонах за 60 днів (2024-02-01 — 2024-03-31), тобто значно більше 100. Повну історію з 60 днів мають 66 зон. 109 зон мають менше 31 дня з поїздками, тож для них `trips_lag_30d` завжди буде `NULL`. Наприклад, зона 1 має лише 10 днів, а зона 2 — один.

## 2. Time-series ML features

Усі вікна мають `PARTITION BY pickup_zone_id`, щоб історія однієї зони не потрапляла в ознаки іншої.

### Feature 1. Lag features

In [6]:
%%sql
SELECT pickup_zone_id, ts_day, trip_count,
       LAG(trip_count, 1) OVER w AS trips_lag_1d,
       LAG(trip_count, 7) OVER w AS trips_lag_7d,
       LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (PARTITION BY pickup_zone_id ORDER BY ts_day)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-02-04,1,None,None,None
1,2024-02-21,2,1,None,None
1,2024-02-25,1,2,None,None
1,2024-03-10,2,1,None,None
1,2024-03-11,1,2,None,None
1,2024-03-14,1,1,None,None
1,2024-03-21,1,1,None,None
1,2024-03-24,1,1,1,None
1,2024-03-25,1,1,2,None
1,2024-03-31,1,1,1,None


**Interpretation:** Перші 20 рядків належать розрідженим зонам 1–3, і на них видно важливе обмеження: `LAG` рахує **рядки**, а не календарні дні. У зоні 1 `trips_lag_1d` для 2024-02-21 бере значення за 2024-02-04, а `trips_lag_7d` уперше з'являється на восьмому рядку (2024-03-24). Для щільних зон із повними 60 днями рядок і день збігаються. У production пропущені дні варто заповнити календарною сіткою (`generate_series` × зони, `trip_count = 0`), щоб lag-1d справді означав «вчора». Нова зона починається з `NULL`, тобто `PARTITION BY` не дає значенням перетікати між зонами.

### Feature 2. Rolling 7-day average fare

In [7]:
%%sql
SELECT pickup_zone_id, ts_day, avg_fare,
       AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
       COUNT(*) OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-02-04,25.00,25.0000000000000000,1
1,2024-02-21,65.35,65.3500000000000000,1
1,2024-02-25,25.00,45.1750000000000000,2
1,2024-03-10,23.00,23.0000000000000000,1
1,2024-03-11,3.00,13.0000000000000000,2
1,2024-03-14,87.40,37.8000000000000000,3
1,2024-03-21,68.10,68.1000000000000000,1
1,2024-03-24,35.00,51.5500000000000000,2
1,2024-03-25,8.60,37.2333333333333333,3
1,2024-03-31,19.00,13.8000000000000000,2


**Interpretation:** На відміну від `LAG`, frame `RANGE BETWEEN INTERVAL '6 days' PRECEDING` рахує календарні дні: для зони 1 на 2024-03-10 у вікні лише 1 день (`days_in_window = 1`), бо попередній запис за 2024-02-25 давніший за 6 днів. На 2024-03-14 у вікно потрапляють 10, 11 і 14 березня (3 дні, середнє 37.80). Тому `days_in_window` варто зберігати поруч з ознакою як показник її надійності.

### Feature 3. Rolling 30-day fare volatility

In [8]:
%%sql
SELECT pickup_zone_id, ts_day, avg_fare,
       STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
       COUNT(*) OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-02-04,25.00,None,1
1,2024-02-21,65.35,28.5317586208771926,2
1,2024-02-25,25.00,23.2960833618013996,3
1,2024-03-10,23.00,23.8943682346559088,3
1,2024-03-11,3.00,26.1361300055434119,4
1,2024-03-14,87.40,34.5310367640474994,5
1,2024-03-21,68.10,32.8418089737253460,6
1,2024-03-24,35.00,31.4451745105668002,6
1,2024-03-25,8.60,31.0982697420994944,7
1,2024-03-31,19.00,31.5230452360234536,7


**Interpretation:** `STDDEV_SAMP` повертає `NULL`, якщо у вікні один рядок: див. перший день кожної зони. Для зони 1 волатильність близько 23–35 через малу кількість поїздок (1–2 на день) — середній тариф там стрибає від 3.00 до 87.40. Отже, на розріджених зонах ця ознака шумна, і її варто використовувати разом із `days_in_window`.

### Feature 4. Expanding cumulative trips

In [9]:
%%sql
SELECT pickup_zone_id, ts_day, trip_count,
       SUM(trip_count) OVER we AS cumulative_trips,
       AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-02-04,1,1,1.00000000000000000000
1,2024-02-21,2,3,1.5000000000000000
1,2024-02-25,1,4,1.3333333333333333
1,2024-03-10,2,6,1.5000000000000000
1,2024-03-11,1,7,1.4000000000000000
1,2024-03-14,1,8,1.3333333333333333
1,2024-03-21,1,9,1.2857142857142857
1,2024-03-24,1,10,1.2500000000000000
1,2024-03-25,1,11,1.2222222222222222
1,2024-03-31,1,12,1.2000000000000000


**Interpretation:** `cumulative_trips` монотонно зростає в межах зони і скидається до `trip_count` на першому дні наступної зони (зона 2: 1). `expanding_avg_trips` — середнє за всю історію до поточного дня включно, тому не містить даних з майбутнього.

### Combined feature view

In [10]:
%%sql
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT pickup_zone_id, ts_day, trip_count, avg_fare, total_distance,
           LAG(trip_count, 1) OVER w_pid AS trips_lag_1d,
           LAG(trip_count, 7) OVER w_pid AS trips_lag_7d,
           LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,
           AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
           STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
           SUM(trip_count) OVER we AS cumulative_trips,
           AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW w_pid AS (PARTITION BY pickup_zone_id ORDER BY ts_day),
           w7 AS (
               PARTITION BY pickup_zone_id
               ORDER BY ts_day::TIMESTAMP
               RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
           ),
           w30 AS (
               PARTITION BY pickup_zone_id
               ORDER BY ts_day::TIMESTAMP
               RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
           ),
           we AS (
               PARTITION BY pickup_zone_id
               ORDER BY ts_day
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
4,2024-03-02,28,20.05,101.06,10,33,10,17.6814285714285714,7.1929159035402533,396,12.7741935483870968
4,2024-03-03,26,18.64,87.29,28,34,12,18.1385714285714286,7.1633824284406284,422,13.1875000000000000
4,2024-03-04,2,28.52,11.11,26,4,23,20.0714285714285714,7.3371332370300483,424,12.8484848484848485
4,2024-03-05,3,8.74,2.62,2,10,23,18.9385714285714286,7.5588099242714370,427,12.5588235294117647
4,2024-03-06,6,19.78,17.40,3,10,4,18.0328571428571429,7.5118187338124162,433,12.3714285714285714
4,2024-03-07,6,14.99,14.07,6,7,4,18.2171428571428571,7.3521089632088936,439,12.1944444444444444
4,2024-03-08,11,19.96,39.98,6,10,8,18.6685714285714286,7.3284642743681063,450,12.1621621621621622
4,2024-03-09,27,20.00,81.79,11,28,5,18.6614285714285714,7.3169925751518695,477,12.5526315789473684
4,2024-03-10,13,16.56,37.36,27,26,12,18.3642857142857143,7.2837398897550697,490,12.5641025641025641
4,2024-03-11,4,19.70,8.38,13,2,23,17.1042857142857143,7.2732944123825623,494,12.3500000000000000


In [11]:
%%sql
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day) AS first_day,
       MAX(ts_day) AS last_day
FROM v_t7_hw_taxi_features;

n_rows,n_zones,first_day,last_day
2902,131,2024-03-02,2024-03-31


**Interpretation:** View містить 2 902 рядки по 131 зоні, від 2024-03-02 до 2024-03-31. Фільтр `trips_lag_30d IS NOT NULL` залишає лише зони, що мають принаймні 31 день із поїздками. Для зони 4 з повною історією перший рядок — 2024-03-02, тобто рівно 30 календарних днів після 2024-02-01. У всіх чотирьох window definitions (`w_pid`, `w7`, `w30`, `we`) є `PARTITION BY pickup_zone_id`.

## 3. JSONB event-log

In [12]:
%%sql
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id   INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id    INTEGER NOT NULL,
    event_ts   TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type TEXT NOT NULL,
    payload    JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT (n % 1000) + 1 AS user_id,
       TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n % 60) || ' days')::INTERVAL AS event_ts,
       CASE (n % 5)
           WHEN 0 THEN 'page_view'
           WHEN 1 THEN 'click'
           WHEN 2 THEN 'add_to_cart'
           WHEN 3 THEN 'checkout'
           ELSE 'recommendation_shown'
       END AS event_type,
       CASE (n % 5)
           WHEN 0 THEN jsonb_build_object(
               'url', '/product/' || ((n % 200) + 1),
               'referrer', (ARRAY['google', 'direct', 'email', 'fb'])[(n % 4) + 1],
               'device', (ARRAY['mobile', 'desktop', 'tablet'])[(n % 3) + 1]
           )
           WHEN 1 THEN jsonb_build_object(
               'element_id', 'btn_' || ((n % 50) + 1),
               'page_url', '/category/' || ((n % 20) + 1),
               'device', (ARRAY['mobile', 'desktop'])[(n % 2) + 1]
           )
           WHEN 2 THEN jsonb_build_object(
               'product_id', (n % 200) + 1,
               'price', ((n % 100) + 10)::NUMERIC(10, 2),
               'quantity', (n % 5) + 1,
               'tags', jsonb_build_array((ARRAY['promo', 'new', 'clearance'])[(n % 3) + 1])
           )
           WHEN 3 THEN jsonb_build_object(
               'order_id', 'o_' || (n + 100000),
               'total', ((n % 500) + 50)::NUMERIC(10, 2),
               'items_count', (n % 10) + 1,
               'payment_method', (ARRAY['card', 'paypal', 'crypto'])[(n % 3) + 1]
           )
           ELSE jsonb_build_object(
               'model_name', 'rec_v' || ((n % 5) + 1),
               'model_version', '1.' || (n % 10),
               'recommendations', jsonb_build_array((n % 200) + 1, ((n + 1) % 200) + 1, ((n + 2) % 200) + 1),
               'experiment', 'exp_' || ((n % 3) + 1)
           )
       END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;

event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


**Interpretation:** 50 000 подій рівномірно розподілені між 5 типами — по 10 000 кожного. Набори ключів `payload` різні: `page_view` має `url`/`referrer`/`device`, `click` — `element_id`/`page_url`/`device`, `add_to_cart` — `product_id`/`price`/`quantity`/`tags`, `checkout` — `order_id`/`total`/`items_count`/`payment_method`, `recommendation_shown` — `model_name`/`model_version`/`recommendations`/`experiment`.

### JSONB Query 1. Containment `@>`

In [13]:
%%sql
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;

event_id,user_id,event_ts,payload
3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


**Interpretation:** `@>` повертає checkout-події, `payload` яких містить пару `payment_method: card`. Через `n % 3` це кожен третій checkout (`event_id` 3, 18, 33, … з кроком 15).

### JSONB Query 2. Key existence `?` та `?|`

In [14]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

event_id,event_type,payload
2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [15]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;

event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


**Interpretation:** `? 'tags'` знаходить лише `add_to_cart`: ключ `tags` є тільки там. `?| ARRAY['experiment', 'model_version']` повертає події, що мають хоча б один із ключів, — тут лише `recommendation_shown`. Обидва оператори перевіряють тільки ключі верхнього рівня.

### JSONB Query 3. JSONPath

In [16]:
%%sql
SELECT event_id, jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

event_id,recommended_pid
4,5
4,6
4,7
9,10
9,11
9,12
14,15
14,16
14,17
19,20


In [17]:
%%sql
SELECT event_id,
       payload ->> 'order_id' AS order_id,
       (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;

event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


**Interpretation:** `jsonb_path_query` розгортає масив `recommendations` у рядки: для кожної події по 3 `product_id`, тому `LIMIT 15` показує 5 подій. `jsonb_path_exists` з фільтром `$.total ? (@ > 300)` повертає checkout-и з `total` від 303.00, а значення порівнюються як числа JSON, без явного cast.

## 4. GIN-index demo + `EXPLAIN ANALYZE`

### Step 1. Containment без GIN

In [18]:
%%sql
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;
ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

QUERY PLAN
Aggregate (cost=1759.05..1759.06 rows=1 width=8) (actual time=30.417..30.418 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=3619 width=0) (actual time=0.008..30.274 rows=3334 loops=1)
"Filter: (payload @> jsonb_build_object('payment_method', 'card'))"
Rows Removed by Filter: 46666
Buffers: shared hit=1000
Planning:
Buffers: shared hit=7
Planning Time: 0.116 ms
Execution Time: 30.434 ms


### Step 2. GIN-індекс `jsonb_path_ops`

In [19]:
%%sql
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);
ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
       pg_size_pretty(pg_relation_size('t7_hw_events')) AS table_size;

gin_index_size,table_size
1144 kB,8000 kB


### Step 3. Той самий containment після GIN

In [20]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

QUERY PLAN
Aggregate (cost=1113.93..1113.94 rows=1 width=8) (actual time=3.846..3.847 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=39.33..1103.28 rows=4263 width=0) (actual time=0.345..3.703 rows=3334 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..38.27 rows=4263 width=0) (actual time=0.237..0.237 rows=3334 loops=1)
"Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))"
Buffers: shared hit=4
Planning:


In [21]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');

QUERY PLAN
Aggregate (cost=1070.25..1070.26 rows=1 width=8) (actual time=2.915..2.916 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=26.08..1063.94 rows=2524 width=0) (actual time=0.291..2.810 rows=2500 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..25.45 rows=2524 width=0) (actual time=0.185..0.185 rows=2500 loops=1)
"Index Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Buffers: shared hit=4
Planning:


**Interpretation:** - **До GIN:** `Seq Scan on t7_hw_events` з `Filter`, перевірено всі 50 000 рядків (`Rows Removed by Filter: 46666`), total cost `1759.06`, 30.4 ms.
- **Після GIN:** `Bitmap Index Scan on idx_t7_events_payload_gin_pathops` → `Bitmap Heap Scan`, total cost `1113.94`, 3.9 ms, той самий результат — 3 334 рядки. Запит із `referrer = google` теж використовує індекс: 2.9 ms, 2 500 рядків.
- Кількість buffers майже не змінилася (1000 → 1004), бо підходящі рядки розкидані по всіх 1000 сторінках таблиці (`Heap Blocks: exact=1000`). Виграш дає саме те, що `@>` не обчислюється для кожного з 50 000 `payload`. Сам індекс займає 1144 kB при таблиці 8000 kB.

### Step 4. Path-equality: без expression index

In [22]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

QUERY PLAN
Aggregate (cost=1750.62..1750.63 rows=1 width=8) (actual time=7.132..7.132 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=250 width=0) (actual time=0.008..6.803 rows=8333 loops=1)
Filter: ((payload ->> 'device'::text) = 'mobile'::text)
Rows Removed by Filter: 41667
Buffers: shared hit=1000
Planning Time: 0.090 ms
Execution Time: 7.144 ms


### Expression B-tree index і повтор запиту

In [23]:
%%sql
CREATE INDEX idx_t7_events_device_btree ON t7_hw_events ((payload ->> 'device'));
ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';

QUERY PLAN
Aggregate (cost=1243.81..1243.82 rows=1 width=8) (actual time=2.313..2.314 rows=1 loops=1)
Buffers: shared hit=1000 read=9
-> Bitmap Heap Scan on t7_hw_events (cost=97.21..1222.87 rows=8377 width=0) (actual time=0.344..1.988 rows=8333 loops=1)
Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Heap Blocks: exact=1000
Buffers: shared hit=1000 read=9
-> Bitmap Index Scan on idx_t7_events_device_btree (cost=0.00..95.12 rows=8377 width=0) (actual time=0.238..0.238 rows=8333 loops=1)
Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Buffers: shared read=9
Planning:


**Interpretation:** - **Без expression index:** знову `Seq Scan` з `Filter: ((payload ->> 'device') = 'mobile')`, 7.1 ms. Planner оцінив `rows=250` замість фактичних 8 333: для довільного виразу без статистики він бере стандартну селективність 0.5%.
- **З `idx_t7_events_device_btree`:** `Bitmap Index Scan` → `Bitmap Heap Scan`, 2.3 ms. Після `ANALYZE` з'явилася статистика виразу, тож оцінка `rows=8377` вже близька до фактичних. GIN `jsonb_path_ops` для цього запиту не використано: він підтримує лише `@>`, `@?` і `@@`, а не порівняння результату `->>`.

## 5. JSONB vs normalized

### Step 1. Нормалізована таблиця checkout-подій

In [24]:
%%sql
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id       INTEGER PRIMARY KEY,
    user_id        INTEGER NOT NULL,
    event_ts       TIMESTAMPTZ NOT NULL,
    order_id       TEXT NOT NULL,
    total          NUMERIC(10, 2) NOT NULL,
    items_count    INTEGER NOT NULL CHECK (items_count > 0),
    payment_method TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (event_id, user_id, event_ts, order_id, total, items_count, payment_method)
SELECT event_id, user_id, event_ts,
       payload ->> 'order_id' AS order_id,
       (payload ->> 'total')::NUMERIC(10, 2) AS total,
       (payload ->> 'items_count')::INTEGER AS items_count,
       payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm ON t7_hw_checkouts_norm (payment_method);
ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm;

n_checkouts
10000


### Step 2. Один запит двома способами

JSONB-версія:

In [25]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT payload ->> 'payment_method' AS payment_method,
       AVG((payload ->> 'total')::NUMERIC) AS avg_total,
       COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;

QUERY PLAN
Limit (cost=1952.87..1952.88 rows=3 width=72) (actual time=8.436..8.437 rows=3 loops=1)
Buffers: shared hit=1000
-> Sort (cost=1952.87..1966.65 rows=5512 width=72) (actual time=8.434..8.435 rows=3 loops=1)
Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=1000
-> HashAggregate (cost=1798.95..1881.63 rows=5512 width=72) (actual time=8.411..8.430 rows=3 loops=1)
Group Key: (payload ->> 'payment_method'::text)
Batches: 1 Memory Usage: 217kB
Buffers: shared hit=1000


Normalized-версія:

In [26]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT payment_method, AVG(total) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

QUERY PLAN
Limit (cost=269.06..269.07 rows=3 width=46) (actual time=2.510..2.512 rows=3 loops=1)
Buffers: shared hit=94
-> Sort (cost=269.06..269.07 rows=3 width=46) (actual time=2.510..2.510 rows=3 loops=1)
Sort Key: (avg(total)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=94
-> HashAggregate (cost=269.00..269.04 rows=3 width=46) (actual time=2.504..2.506 rows=3 loops=1)
Group Key: payment_method
Batches: 1 Memory Usage: 24kB
Buffers: shared hit=94


Сам результат (однаковий для обох версій):

In [27]:
%%sql
SELECT payment_method, ROUND(AVG(total), 2) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

payment_method,avg_total,n
paypal,300.52,3333
card,300.50,3334
crypto,300.48,3333


**Interpretation:** Обидві версії дають однаковий результат: середній `total` майже однаковий (300.48–300.52) з 3 333–3 334 checkout-ами на метод, бо дані синтетичні та рівномірні.
- **JSONB:** `Seq Scan on t7_hw_events` з `Filter: event_type = 'checkout'` (`Rows Removed by Filter: 40000`) → `HashAggregate`, 1000 buffers, 8.5 ms. Planner очікував `rows=5512` груп, бо не знає, скільки різних значень `payload ->> 'payment_method'`.
- **Normalized:** `Seq Scan on t7_hw_checkouts_norm` → `HashAggregate`, 94 buffers, 2.5 ms, точна оцінка — 3 групи.
- Виграш нормалізованої таблиці в тому, що вона в ~10 разів менша (тільки checkout і вузькі typed-колонки), а не в індексі: для агрегації по всіх рядках обидва plans читають таблицю повністю.

### Bonus. MongoDB-equivalent (pseudocode)

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL
db.events.aggregate([
  { $match: { event_type: "checkout" } },
  { $group: {
      _id: "$payload.payment_method",
      avg_total: { $avg: { $toDouble: "$payload.total" } },
      n: { $sum: 1 }
  } },
  { $sort: { avg_total: -1 } },
  { $limit: 3 }
]);
```

- PostgreSQL JSONB зручний, коли потрібні SQL, `JOIN`, транзакції та зв'язок з нормалізованими таблицями: тут той самий `payload` можна з'єднати з `t7_hw_checkouts_norm` або таблицею користувачів в одному запиті.
- MongoDB aggregation pipeline природний для document-first workloads, де документ — основна одиниця, і `$match → $group → $sort` відповідає `WHERE → GROUP BY → ORDER BY`.
- High-volume raw clickstream (сотні мільйонів `page_view`/`click`) може краще жити в document store або data lake (Parquet), де дешевий запис і масштабування.
- Curated ML features (як `v_t7_hw_taxi_features` або `t7_hw_checkouts_norm`) зручніше тримати в typed SQL-таблицях: типи, `CHECK` і стабільна схема для training/serving.

## Reflection

**1. Плани до і після GIN.** До індексу containment `@>` виконувався через `Seq Scan` (total cost `1759.06`, 30.4 ms) і перевіряв усі 50 000 рядків. Після `jsonb_path_ops` planner обрав `Bitmap Index Scan` + `Bitmap Heap Scan` (total cost `1113.94`, 3.9 ms). Buffers майже не змінилися (1000 → 1004), бо 3 334 рядки розкидані по всіх сторінках, тож виграш дає відсутність перевірки `@>` для кожного `payload`.

**2. `@>` проти `->>`.** GIN індексує ключі та значення всередині документа і підтримує оператори `@>`, `@?`, `@@`. Вираз `payload ->> 'device' = 'mobile'` спочатку обчислює текст, а потім порівнює його через `=`, і GIN цього не розуміє: план залишився `Seq Scan` (7.1 ms, оцінка `rows=250` замість 8 333). Expression B-tree дав `Bitmap Index Scan` (2.3 ms) і точнішу оцінку. Альтернатива — переписати умову як `payload @> '{"device":"mobile"}'`.

**3. Read/write trade-off.** Пришвидшуються containment і JSONPath-фільтри. Натомість кожен `INSERT`/`UPDATE` оновлює багато GIN-ключів (індекс займає 1144 kB при таблиці 8000 kB), що робить запис і `VACUUM` дорожчими. Тому кожен індекс має відповідати реальному запиту.

**4. JSONB чи normalized.** JSONB підходить для raw event-log, де схема payload змінюється і кожен тип має свої поля. Нормалізована таблиця — для curated features: агрегація займала 2.5 ms проти 8.5 ms і читала 94 buffers проти 1000, а `CHECK` гарантує валідний `payment_method`.

**5. Новий `refund`.** У JSONB достатньо вставляти `event_type = 'refund'` з потрібними ключами (`order_id`, `amount`, `reason`) — без DDL, лише оновити валідацію в ETL і, можливо, додати partial index. У normalized-підході потрібна міграція: `CREATE TABLE t7_hw_refunds_norm` з FK на `order_id`, `CHECK (amount > 0)`, backfill з event-log і оновлення feature-запитів.